# Воспроизводимость решения

In [1]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Загрузка данных

In [2]:
import pandas as pd

In [3]:
articles = pd.read_feather("candidate_data/articles.f")
calibration = pd.read_feather("candidate_data/calibration.f")
test = pd.read_feather("candidate_data/test.f")

d:\Глеб\Pet-projects\avito_test-task\.venv\Lib\site-packages\pandas\io\feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [4]:
articles.head()

,article_id,title,body
0,1730,Имя или название компании,"<ol><li><p>Зайдите в раздел <a href=""https://w..."
1,1746,"Понять, что профиль заблокирован","<p>Проверьте, какое сообщение вы видите при вх..."
2,1747,Не допустить блокировки профиля,<ol><li><p><strong>Не заводите несколько аккау...
3,1774,Оставить или удалить профиль,<p>⚡ Не удаляйте профиль с подтверждёнными дан...
4,1775,Удалить профиль,"<p>⚡ Удалить профиль не получится, если у вас ..."


In [5]:
calibration.head()

,query_id,query_text,ground_truth
0,1,Как передать товар через службу авито,1909 4234
1,2,"Можете подсказать, если заказать товар Авито д...",2865 4400
2,3,Здравствуйте. Как отправить товар через Авито.,1909
3,4,как получить деньги за возрат если продавец уж...,4400 4403
4,5,"Когда мне прийдут деньги за доставку, сегодня ...",4361


In [6]:
test.head()

,query_id,query_text
0,1,"Здравствуйте! Подскажите, пожалуйста, не могу ..."
1,2,"Здравствуйте , почему так долго доставляется в..."
2,3,"Здравствуйте,подскажите как мне отправить крос..."
3,4,Здравствуйте! В каких случаях за возврат снима...
4,5,Почему у меня доставки в несколько раз дороже ...


# EDA

In [7]:
print(f"Articles: {len(articles)}")
print(f"Calibration: {len(calibration)}")
print(f"Test: {len(test)}")

Articles: 793
Calibration: 500
Test: 500


In [8]:
articles.isna().sum()

article_id    0
title         0
body          0
dtype: int64

In [9]:
articles.sample(5)

,article_id,title,body
137,2242,Несколько телефонов,<p><strong>Почему отклонили:</strong> в объявл...
198,2795,Почему объявление скрыли,<p>Если объявление с квартирой или домом скрыт...
739,4447,Настроить рассылку скидок,"<p>Гостям, которые интересовались вашим объявл..."
583,4278,Подписка на инструменты «Мастер услуг»,"<headline name=""Как работает подписка на инстр..."
655,4356,Ошибки при автозагрузке,"<headline name=""Автозагрузка: ошибки в объявле..."


# Предобработка текста

## Удаление эмодзи, html разметки, приведение к нижнему регистру, нормализация пробелов

In [10]:
import re

EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F" 
    "\U0001F300-\U0001F5FF"  
    "\U0001F680-\U0001F6FF" 
    "\U0001F1E0-\U0001F1FF" 
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)

In [11]:
from bs4 import BeautifulSoup

def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(" ", strip=True)
    return EMOJI_PATTERN.sub("", text)

In [12]:
articles["body_clean"] = articles.body.apply(clean_html)

articles["text"] = ( 3*(articles["title"]+ " ")
    + articles["body_clean"]
)

In [13]:
def normalize(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.replace("ё", "е")
    text = text.replace("\xa0", " ")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [14]:
articles["text"] = (
    articles["text"]
    .apply(normalize)
)

In [15]:
articles.head()

,article_id,title,body,body_clean,text
0,1730,Имя или название компании,"<ol><li><p>Зайдите в раздел <a href=""https://w...",Зайдите в раздел Управление профилем . Нажмите...,имя или название компании имя или название ком...
1,1746,"Понять, что профиль заблокирован","<p>Проверьте, какое сообщение вы видите при вх...","Проверьте, какое сообщение вы видите при входе...","понять, что профиль заблокирован понять, что п..."
2,1747,Не допустить блокировки профиля,<ol><li><p><strong>Не заводите несколько аккау...,Не заводите несколько аккаунтов для продаж в о...,не допустить блокировки профиля не допустить б...
3,1774,Оставить или удалить профиль,<p>⚡ Не удаляйте профиль с подтверждёнными дан...,Не удаляйте профиль с подтверждёнными данными...,оставить или удалить профиль оставить или удал...
4,1775,Удалить профиль,"<p>⚡ Удалить профиль не получится, если у вас ...","Удалить профиль не получится, если у вас есть...",удалить профиль удалить профиль удалить профил...


# Chunking

Разбиваем длинные статьи на блоки по 200 слов с шагом 30 слов, чтобы улучшить семантический поиск

In [16]:
from tqdm.auto import tqdm


def split_into_chunks(
    text,
    chunk_size=200,
    overlap=30
):
    
    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = words[start:end]

        chunks.append(
            " ".join(chunk)
        )

        start += chunk_size - overlap

    return chunks



chunks_data = []


for _, row in tqdm(
    articles.iterrows(),
    total=len(articles),
    desc="Creating chunks"
):

    chunks = split_into_chunks(
        row.text,
        chunk_size=200,
        overlap=30
    )


    for chunk_id, chunk in enumerate(chunks):

        chunks_data.append(
            {
                "article_id": row.article_id,
                "chunk_id": chunk_id,
                "text": chunk
            }
        )


chunks = pd.DataFrame(chunks_data)


print(
    "Статей:",
    len(articles)
)

print(
    "Chunks:",
    len(chunks)
)

Creating chunks:   0%|          | 0/793 [00:00<?, ?it/s]

Статей: 793
Chunks: 3896


In [17]:
chunks.head()

,article_id,chunk_id,text
0,1730,0,имя или название компании имя или название ком...
1,1746,0,"понять, что профиль заблокирован понять, что п..."
2,1746,1,создать новый профиль — с новым номером телефо...
3,1746,2,попросим подтвердить его перед связкой. после ...
4,1746,3,"проверка, в новом ее проходить нельзя — иначе ..."


# Модели

## BM25

Используем rank_bm25 с лемматизацией (pymorphy3) и удалением стоп-слов с помощью nltk

In [18]:
import nltk
from nltk.corpus import stopwords
import pymorphy3

nltk.download('stopwords')

morph = pymorphy3.MorphAnalyzer()
russian_stopwords = set(stopwords.words('russian'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [19]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    lemmatized = []
    for token in tokens:
        if token not in russian_stopwords:
            lemma = morph.parse(token)[0].normal_form
            lemmatized.append(lemma)
    return lemmatized

In [20]:
from rank_bm25 import BM25Okapi

bm25_corpus = [
    tokenize(text)
    for text in chunks.text
]


bm25 = BM25Okapi(
    bm25_corpus
)

## Эмбеддинги (multilingual-e5-large)

Модель `intfloat/multilingual-e5-large` (560M параметров) генерирует векторные представления для чанков. Для запросов используется префикс `query:`, для документов – `passage:` согласно рекомендациям авторов.

In [21]:
def prepare_query(text):
    return "query: " + text


def prepare_document(text):
    return "passage: " + text

In [22]:
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "intfloat/multilingual-e5-large",
    device= device
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [23]:
documents = [
    prepare_document(text)
    for text in tqdm(
        chunks.text.tolist(),
        desc="Preparing chunks"
    )
]


doc_embeddings = model.encode(
    documents,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

Preparing chunks:   0%|          | 0/3896 [00:00<?, ?it/s]

Batches:   0%|          | 0/122 [00:00<?, ?it/s]

## Индексация в FAISS

Строим плоский индекс с косинусным сходством

In [24]:
import faiss

embedding_dim = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

index.add(
    np.asarray(doc_embeddings).astype("float32")
)

print("Документов в FAISS:", index.ntotal)

Документов в FAISS: 3896


## Похожесть заголовка

Вычисляется TF-IDF косинусная близость между запросом и заголовком каждой статьи. Используем при переранжировании

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


title_vectorizer = TfidfVectorizer(
    ngram_range=(1,2)
)


title_matrix = title_vectorizer.fit_transform(
    articles.title.fillna("")
)

In [26]:
def get_title_score(query, article_ids):

    query_vector = title_vectorizer.transform(
        [query]
    )


    indexes = []

    for aid in article_ids:

        idx = articles.index[
            articles.article_id == aid
        ][0]

        indexes.append(idx)


    scores = cosine_similarity(
        query_vector,
        title_matrix[indexes]
    )[0]


    return scores

## Поиск кандидатов

### Топ кандидатов от bm25

In [27]:
def bm25_search(query, k=100):

    tokens = tokenize(query)

    scores = bm25.get_scores(tokens)

    top_ids = np.argsort(scores)[::-1][:k]

    return [
        {
            "article_id": chunks.iloc[i].article_id,
            "score": scores[i],
            "rank": rank + 1
        }
        for rank, i in enumerate(top_ids)
    ]

### Топ кандидатов от e5

In [28]:
def dense_search(query, k=100):

    query_embedding = model.encode(
        [
            prepare_query(query)
        ],
        normalize_embeddings=True
    )

    scores, ids = index.search(
        np.asarray(query_embedding).astype("float32"),
        k
    )


    result = []

    for rank, idx in enumerate(ids[0]):

        result.append(
            {
                "article_id": chunks.iloc[idx].article_id,
                "score": float(scores[0][rank]),
                "rank": rank + 1
            }
        )

    return result

## Объединение двух моделей

$$
RRF = 1/ (5+rank)
$$

Используется Reciprocal Rank Fusion с параметром k=5, что усиливает вклад первых позиций. Веса BM25 и dense равны (alpha=1.0).

In [128]:
from collections import defaultdict


def rrf_merge(
    bm25_results,
    dense_results,
    k=5,
    alpha = 1.0
):

    scores = defaultdict(float)


    for item in bm25_results:

        scores[item["article_id"]] += alpha * (
            1 / (k + item["rank"])
        )


    for item in dense_results:

        scores[item["article_id"]] += (
            1 / (k + item["rank"])
        )


    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )


    return [
        {
            "article_id": article_id,
            "rrf_score": score
        }
        for article_id, score in ranked
    ]

## Переранжирование с учётом заголовка

Нормализуем RRF-оценки и оценки похожести заголовка (min‑max), затем складываем с весом 0.1 для заголовка. Сортируем по итоговой оценке

In [129]:
def minmax(values):

    values = np.array(values)

    if values.max() == values.min():
        return np.zeros_like(values)

    return (
        values - values.min()
    ) / (
        values.max() - values.min()
    )

In [130]:
def rerank(query, candidates):

    article_ids = [
        x["article_id"]
        for x in candidates
    ]


    title_scores = get_title_score(
        query,
        article_ids
    )


    rrf_scores = [
        x["rrf_score"]
        for x in candidates
    ]


    rrf_scores = minmax(
        rrf_scores
    )


    title_scores = minmax(
        title_scores
    )


    for i, item in enumerate(candidates):

        item["final_score"] = (
            rrf_scores[i]
            +
            0.1 * title_scores[i]
        )


    return sorted(
        candidates,
        key=lambda x: x["final_score"],
        reverse=True
    )

## Функция поиска

In [131]:
def retrieve(query, top_k=10):

    query = normalize(query)


    bm25_candidates = bm25_search(
        query,
        k=200
    )


    dense_candidates = dense_search(
        query,
        k=200
    )


    candidates = rrf_merge(
        bm25_candidates,
        dense_candidates
    )


    candidates = rerank(
        query,
        candidates[:100]
    )


    results = []
    seen = set()


    for x in candidates:

        article_id = int(
            x["article_id"]
        )

        if article_id not in seen:

            results.append(
                article_id
            )

            seen.add(
                article_id
            )


        if len(results) == top_k:
            break


    return results

### Пример

In [132]:
retrieve(
    "как отправить товар покупателю"
)

[1909, 4234, 4387, 4286, 4409, 4407, 1918, 4400, 4328, 4308]

In [133]:
articles[articles['article_id'] == retrieve(
    "как отправить товар покупателю"
)[0]]

,article_id,title,body,body_clean,text
48,1909,Отправить заказ,"<ol><li><p>У вас будет <strong>2 рабочих дня, ...","У вас будет 2 рабочих дня, чтобы отправить тов...",отправить заказ отправить заказ отправить зака...


# Оценка решения на калибровочных данных

Считаются метрики MAP@10 и Recall@10

In [134]:
def average_precision_at_k(
    predicted,
    actual,
    k=10
):

    actual = set(actual)

    predicted = predicted[:k]


    score = 0
    hits = 0


    for i, article_id in enumerate(
        predicted,
        start=1
    ):

        if article_id in actual:

            hits += 1

            score += hits / i


    if len(actual) == 0:
        return 0


    return score / min(
        len(actual),
        k
    )

def evaluate_map10(df):

    scores = []


    for _, row in df.iterrows():

        prediction = retrieve(
            row.query_text,
            top_k=10
        )


        target = list(
            map(
                int,
                row.ground_truth.split()
            )
        )


        score = average_precision_at_k(
            prediction,
            target,
            k=10
        )


        scores.append(score)


    return np.mean(scores)

In [135]:
def recall_at_k(predicted, actual, k=10):

    predicted = set(predicted[:k])
    actual = set(actual)

    return len(predicted & actual) / len(actual)

def evaluate_recall10(df):

    scores = []

    for _, row in df.iterrows():

        prediction = retrieve(
            row.query_text,
            top_k=10
        )

        target = list(
            map(
                int,
                row.ground_truth.split()
            )
        )

        scores.append(
            recall_at_k(
                prediction,
                target,
                10
            )
        )

    return np.mean(scores)

In [136]:
map10 = evaluate_map10(calibration)
print(f"MAP@10: {map10:.4f}")
recall10 = evaluate_recall10(calibration)
print(f"Recall@10: {recall10:.4f}")

MAP@10: 0.4889
Recall@10: 0.7923


# Предсказания на тесте

Выполняется генерация предсказаний для тестовой выборки, а также сохранение результата в csv файл

In [137]:
test_predictions = []


for query in test.query_text:

    result = retrieve(
        query,
        top_k=10
    )

    test_predictions.append(
        " ".join(
            map(str, result)
        )
    )

In [138]:
answer = pd.DataFrame(
    {
        "query_id": test.query_id,
        "answer": test_predictions
    }
)

In [139]:
answer.head()

,query_id,answer
0,1,3565 2196 4308 4257 2962 1960 4387 4234 4409 2964
1,2,4408 4234 4009 2408 4400 3209 4407 4396 1923 3055
2,3,4234 1909 4308 4396 4400 4408 4219 4362 4387 4286
3,4,4234 4400 2865 4408 4308 2646 4219 4361 4532 4331
4,5,4308 4294 4409 4249 4321 2232 4320 4436 3088 4258


In [140]:
answer.to_csv(
    "answer.csv",
    index=False
)